In [34]:
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================================
# ФУНКЦИИ ЧТЕНИЯ ДАННЫХ (без изменений)
# ============================================================================
def read_output_characteristics(filename):
    abs_path = os.path.abspath(filename)
    if not os.path.exists(abs_path):
        print(f"❌ Файл '{filename}' не найден")
        return {}
    td = {}
    current_idx = None
    uke_vals, ik_vals = [], []
    with open(abs_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if 'AM1[' in line:
                if current_idx is not None and uke_vals:
                    td[current_idx] = (np.array(uke_vals), np.array(ik_vals))
                try:
                    idx = int(line.split('AM1[')[1].split(']')[0])
                    current_idx = idx
                except:
                    current_idx = len(td)
                uke_vals, ik_vals = [], []
            else:
                try:
                    parts = line.split()
                    if len(parts) >= 2:
                        uke_vals.append(float(parts[0].replace(',', '.')))
                        ik_vals.append(float(parts[1].replace(',', '.')))
                except: continue
        if current_idx is not None and uke_vals:
            td[current_idx] = (np.array(uke_vals), np.array(ik_vals))
    if not td:
        print("⚠️  Данные не распознаны")
        return {}
    try:
        ib_min = float(input("Мин. ток базы (мкА) [30]: ") or 30)
        ib_max = float(input("Макс. ток базы (мкА) [130]: ") or 130)
        if ib_max < ib_min: ib_min, ib_max = ib_max, ib_min
    except:
        ib_min, ib_max = 20, 80
    sorted_idx = sorted(td.keys())
    n = len(sorted_idx)
    data = {}
    if n == 1:
        data[round((ib_min+ib_max)/2, 2)] = td[sorted_idx[0]]
    else:
        step = (ib_max - ib_min) / (n - 1)
        for i, idx in enumerate(sorted_idx):
            data[round(ib_min + i*step, 2)] = td[idx]
    return data

def read_input_characteristic(filename):
    abs_path = os.path.abspath(filename)
    if not os.path.exists(abs_path):
        return None, None
    ube_vals, ib_vals = [], []
    with open(abs_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or 'AM1' in line: continue
            try:
                parts = line.split()
                if len(parts) >= 2:
                    ube = float(parts[0].replace(',', '.'))
                    ib = float(parts[1].replace(',', '.'))
                    if ib < 1: ib *= 1e6
                    ube_vals.append(ube)
                    ib_vals.append(ib)
            except: continue
    if len(ube_vals) < 3: return None, None
    return np.array(ube_vals), np.array(ib_vals)

def find_intersection_single(uke_curve, ik_curve, E_k, R_k):
    ik_load = (E_k - uke_curve) / R_k * 1000
    diff = ik_curve * 1000 - ik_load
    for i in range(len(diff) - 1):
        if diff[i] == 0:
            return uke_curve[i], ik_curve[i] * 1000
        elif diff[i] * diff[i+1] < 0:
            t = abs(diff[i]) / (abs(diff[i]) + abs(diff[i+1]))
            uke_int = uke_curve[i] + t * (uke_curve[i+1] - uke_curve[i])
            return uke_int, (E_k - uke_int) / R_k * 1000
    if len(diff) > 0 and diff[0] == 0:
        return uke_curve[0], ik_curve[0] * 1000
    if len(diff) > 0 and diff[-1] == 0:
        return uke_curve[-1], ik_curve[-1] * 1000
    return None

def build_transfer_characteristic(data, E_k, R_k):
    ib_vals, ik_vals = [], []
    for ib, (uke, ik) in sorted(data.items()):
        result = find_intersection_single(uke, ik, E_k, R_k)
        if result:
            ib_vals.append(ib)
            ik_vals.append(result[1])
    return (np.array(ib_vals), np.array(ik_vals)) if ib_vals else (None, None)

# ============================================================================
# НОВЫЕ ФУНКЦИИ: РАСЧЁТ ПАРАМЕТРОВ УСИЛИТЕЛЯ
# ============================================================================
def calculate_amplifier_params(ib0_ua, ik0_ma, uke0_v, ube0_v, R_k_ohm):
    """
    Расчёт параметров усилительного каскада с ОЭ (разделы 3.4 и 3.5 методички)
    Возвращает словарь с рассчитанными значениями
    """
    params = {}
    
    # === 3.4.1: Цепь термостабилизации ===
    # Выбираем RЭ ≈ (0.05...0.15) * Rк
    params['R_E_factor'] = 0.1  # коэффициент в диапазоне 0.05...0.15
    params['R_E'] = params['R_E_factor'] * R_k_ohm  # Ом
    
    # Корректировка Eк по уравнению: Eк = Uкэ0 + (Rк + Rэ)*Iк0
    ik0_a = ik0_ma / 1000  # перевод в А
    params['E_K_corrected'] = uke0_v + (R_k_ohm + params['R_E']) * ik0_a
    
    # Расчёт Cэ: Cэ = 10^7 / (k * 2π * fн * Rэ), мкФ
    params['f_low'] = 100  # нижняя граничная частота, Гц
    params['C_E_factor'] = 1.5  # коэффициент в диапазоне 1...2
    params['C_E'] = 1e7 / (params['C_E_factor'] * 2 * np.pi * params['f_low'] * params['R_E'])  # мкФ
    
    # === 3.4.2: Делитель смещения ===
    # Оцениваем Rвх.тр ≈ 1...5 кОм (типично для малосигнальных БТ)
    params['R_in_transistor'] = 2000  # Ом, можно уточнить по входной характеристике
    
    # Rб = R1||R2 = (2...5) * Rвх.тр
    params['R_B_factor'] = 3
    params['R_B'] = params['R_B_factor'] * params['R_in_transistor']
    
    # Iдел = (2...5) * Iб0
    params['I_div_factor'] = 3
    params['I_div'] = params['I_div_factor'] * ib0_ua  # мкА
    
    # Расчёт R1, R2
    ib0_a = ib0_ua / 1e6
    params['R1'] = (params['E_K_corrected'] - ube0_v) / (params['I_div']/1e6 + ib0_a)  # Ом
    params['R2'] = (params['R_E'] * ik0_a + ube0_v) / (params['I_div']/1e6)  # Ом
    
    # Проверка: Rб = R1*R2/(R1+R2)
    params['R_B_calc'] = (params['R1'] * params['R2']) / (params['R1'] + params['R2'])
    
    # === 3.4.3: Разделительный конденсатор ===
    params['R_in'] = (params['R_B'] * params['R_in_transistor']) / (params['R_B'] + params['R_in_transistor'])
    params['C_R1'] = 1e7 / (params['C_E_factor'] * 2 * np.pi * params['f_low'] * params['R_in'])  # мкФ
    
    # === 3.5: Параметры усилителя ===
    # Оцениваем β = h21э ≈ ΔIк/ΔIб из переходной характеристики
    params['beta'] = 100  # типичное значение, можно уточнить по данным
    
    # Коэффициент усиления по току
    params['K_i'] = params['beta']
    
    # Входное сопротивление каскада
    params['R_in_stage'] = params['R_in']
    
    # Выходное сопротивление (приближённо)
    params['R_out_stage'] = R_k_ohm  # если h22*Rк << 1
    
    # Коэффициент усиления по напряжению (с учётом инверсии)
    params['K_u'] = -params['beta'] * R_k_ohm / params['R_in_stage']
    
    # Коэффициент усиления по мощности
    params['K_p'] = params['K_i'] * abs(params['K_u'])
    
    # Полезная выходная мощность (при синусоидальном сигнале)
    uke_m_v = (ik0_ma * params['R_E_factor']) * R_k_ohm / 1000  # оценка амплитуды
    params['P_out'] = 0.5 * (uke_m_v ** 2) / R_k_ohm  # Вт
    
    # Полная мощность от источника
    params['P_supply'] = ik0_a * params['E_K_corrected'] + (params['I_div']/1e6)**2 * (params['R1'] + params['R2']) + ib0_a**2 * params['R1']
    
    # КПД
    params['eta'] = params['P_out'] / params['P_supply'] * 100 if params['P_supply'] > 0 else 0
    
    # Частотные параметры (оценочно)
    params['C_K'] = 5e-12  # ёмкость коллекторного перехода, Ф (типично)
    params['tau_low'] = 1e-6 * (params['R_in_stage'] + params['R_out_stage'])  # оценка с Cр1
    params['tau_high'] = params['C_K'] * (params['R_in_stage'] * params['R_out_stage']) / (params['R_in_stage'] + params['R_out_stage'])
    params['f_high'] = 1 / (2 * np.pi * params['tau_high']) if params['tau_high'] > 0 else 1e6
    
    return params

# ============================================================================
# НОВАЯ ФУНКЦИЯ: ПОСТРОЕНИЕ ВРЕМЕННЫ́Х ДИАГРАММ
# ============================================================================
def plot_time_diagrams(ib0, ib_m, ube0, ube_m, ik0, ik_m, uke0, uke_m, f_signal=1000):
    """
    Построение временны́х зависимостей токов и напряжений
    """
    # Параметры сигнала
    t_period = 1 / f_signal
    t = np.linspace(0, 2*t_period, 1000)  # два периода
    omega = 2 * np.pi * f_signal
    
    # Сигналы (с инверсией для uкэ в схеме с ОЭ)
    i_b = ib0 + ib_m * np.sin(omega * t)  # мкА
    u_be = ube0 + ube_m * np.sin(omega * t)  # В
    i_c = ik0 + ik_m * np.sin(omega * t)  # мА
    u_ce = uke0 - uke_m * np.sin(omega * t)  # В (инверсия!)
    
    fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
    
    # iБ(t)
    axes[0].plot(t*1000, i_b, 'b-', linewidth=1.5)
    axes[0].axhline(y=ib0, color='gray', linestyle=':', linewidth=0.5)
    axes[0].axhline(y=ib0+ib_m, color='blue', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[0].axhline(y=ib0-ib_m, color='blue', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[0].set_ylabel('iБ, мкА', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title('Временны́е диаграммы сигналов', fontweight='bold')
    
    # uБЭ(t)
    axes[1].plot(t*1000, u_be, 'b-', linewidth=1.5)
    axes[1].axhline(y=ube0, color='gray', linestyle=':', linewidth=0.5)
    axes[1].axhline(y=ube0+ube_m, color='blue', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[1].axhline(y=ube0-ube_m, color='blue', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[1].set_ylabel('uБЭ, В', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # iК(t)
    axes[2].plot(t*1000, i_c, 'r-', linewidth=1.5)
    axes[2].axhline(y=ik0, color='gray', linestyle=':', linewidth=0.5)
    axes[2].axhline(y=ik0+ik_m, color='red', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[2].axhline(y=ik0-ik_m, color='red', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[2].set_ylabel('iК, мА', fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    
    # uКЭ(t) — с инверсией
    axes[3].plot(t*1000, u_ce, 'r-', linewidth=1.5)
    axes[3].axhline(y=uke0, color='gray', linestyle=':', linewidth=0.5)
    axes[3].axhline(y=uke0+uke_m, color='red', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[3].axhline(y=uke0-uke_m, color='red', linestyle=':', linewidth=0.5, alpha=0.5)
    axes[3].set_ylabel('uКЭ, В', fontweight='bold')
    axes[3].set_xlabel('Время, мс', fontweight='bold')
    axes[3].grid(True, alpha=0.3)
    
    # Подписи амплитуд
    for ax, val, m, label in zip(axes, [ib0, ube0, ik0, uke0], [ib_m, ube_m, ik_m, uke_m], ['мкА', 'В', 'мА', 'В']):
        ax.text(t_period*1.02*1000, val, f'{label}: {val:.2f}±{m:.2f}{val}', 
               fontsize=8, va='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    plt.tight_layout()
    plt.savefig('04_time_diagrams.png', dpi=300, bbox_inches='tight')
    print("✅ Сохранён: 04_time_diagrams.png")
    plt.close()

# ============================================================================
# ОСНОВНОЙ БЛОК
# ============================================================================
print("🔧 Расчёт усилительного каскада с ОЭ")

# Загрузка данных
od = read_output_characteristics('tcurve.txt')
ube_data, ib_in_data = read_input_characteristic('tcurve_вх_2.txt')

if not od:
    print("❌ Нет данных для расчёта")
else:
    print(f"✅ Загружено кривых: {len(od)}")
    
    # Параметры транзистора и схемы
    U_ke_max, I_k_max, P_k_max = 50, 50, 0.625
    try:
        E_k = float(input(f"Eк [40]: ") or 40)
        R_k = float(input(f"Rк [кОм, 1.5]: ") or 1.5) * 1000
        ib_target = float(input(f"Ток базы для рабочей точки [40]: ") or 40)
    except:
        E_k, R_k, ib_target = 40, 1500, 40
    
    # Выбор рабочей точки
    available_ib = sorted(od.keys())
    ib_closest = min(available_ib, key=lambda x: abs(x - ib_target))
    uke_curve, ik_curve = od[ib_closest]
    q_point = find_intersection_single(uke_curve, ik_curve, E_k, R_k)
    
    if q_point is None:
        print("❌ Пересечение не найдено")
    else:
        uke_q, ik_q = q_point
        ube_q = np.interp(ib_closest, ib_in_data, ube_data) if ube_data is not None else 0.7
        
        # Расчёт амплитуд
        ib_trans, ik_trans = build_transfer_characteristic(od, E_k, R_k)
        ib_m = ik_m = uke_m = ube_m = 0
        if ib_trans is not None and len(ib_trans) >= 3:
            idx_q = np.argmin(np.abs(ib_trans - ib_closest))
            idx_min, idx_max = max(0, idx_q-1), min(len(ib_trans)-1, idx_q+1)
            ib_m = min(ib_closest - ib_trans[idx_min], ib_trans[idx_max] - ib_closest)
            ik_m = min(ik_q - ik_trans[idx_min], ik_trans[idx_max] - ik_q)
            uke_m = ik_m * R_k / 1000
            if ube_data is not None:
                ube_low = np.interp(ib_closest - ib_m, ib_in_data, ube_data)
                ube_high = np.interp(ib_closest + ib_m, ib_in_data, ube_data)
                ube_m = min(ube_q - ube_low, ube_high - ube_q)
            else:
                ube_m = 0.02
        
        ib_pp, ik_pp, uke_pp, ube_pp = 2*ib_m, 2*ik_m, 2*uke_m, 2*ube_m
        
        print(f"\n🎯 Рабочая точка (режим покоя):")
        print(f"   IБ0 = {ib_closest:.2f} мкА")
        print(f"   IК0 = {ik_q:.2f} мА")
        print(f"   UБЭ0 = {ube_q:.3f} В")
        print(f"   UКЭ0 = {uke_q:.2f} В")
        
        # === РАСЧЁТ ПАРАМЕТРОВ УСИЛИТЕЛЯ (разделы 3.4, 3.5) ===
        print(f"\n⚙️  Расчёт параметров усилителя...")
        params = calculate_amplifier_params(ib_closest, ik_q, uke_q, ube_q, R_k)
        
        print(f"\n📋 Элементы схемы:")
        print(f"   Rэ = {params['R_E']:.0f} Ом  (падение ≈ {params['R_E_factor']*100:.0f}% от Eк)")
        print(f"   Cэ = {params['C_E']:.1f} мкФ  (fн = {params['f_low']} Гц)")
        print(f"   R1 = {params['R1']/1000:.2f} кОм")
        print(f"   R2 = {params['R2']/1000:.2f} кОм")
        print(f"   Rб = {params['R_B_calc']/1000:.2f} кОм")
        print(f"   Cр1 = {params['C_R1']:.2f} мкФ")
        print(f"   Eк (скорр.) = {params['E_K_corrected']:.1f} В")
        
        print(f"\n📊 Параметры усилителя:")
        print(f"   β (оценка) = {params['beta']}")
        print(f"   Kᵢ = {params['K_i']:.1f}  (по току)")
        print(f"   Kᵤ = {params['K_u']:.1f}  (по напряжению, знак «-» = инверсия)")
        print(f"   Kₚ = {params['K_p']:.0f}  (по мощности)")
        print(f"   Rвх = {params['R_in_stage']/1000:.2f} кОм")
        print(f"   Rвых ≈ {params['R_out_stage']/1000:.2f} кОм")
        print(f"   Pвых = {params['P_out']*1000:.2f} мВт")
        print(f"   Pпит = {params['P_supply']*1000:.2f} мВт")
        print(f"   η = {params['eta']:.1f}%")
        print(f"   fверх ≈ {params['f_high']/1e6:.2f} МГц")
        
        # === ГРАФИКИ ХАРАКТЕРИСТИК (как раньше, с исправлениями) ===
        IK_MIN, IK_MAX = 0, I_k_max * 1.1
        IB_MIN, IB_MAX = 0, max(max(od.keys()) * 1.2, 100)
        
        # График 1: выходные характеристики
        fig_out = plt.figure(figsize=(10, 7))
        ax_out = fig_out.add_subplot(1, 1, 1)
        colors = plt.cm.viridis(np.linspace(0, 1, len(od)))
        for idx, (ib, (uke, ik)) in enumerate(sorted(od.items())):
            ax_out.plot(uke, ik*1000, color=colors[idx], linewidth=1.2, label=f'Iб={ib:.0f}мкА', alpha=0.7)
        ax_out.plot([E_k, 0], [0, E_k/R_k*1000], 'r--', linewidth=2.5, label='Нагрузочная прямая')
        ax_out.axvline(x=U_ke_max, color='magenta', linestyle='-.', linewidth=1.5, label='Uкэ max')
        ax_out.axhline(y=I_k_max, color='magenta', linestyle='-.', linewidth=1.5, label='Iк max')
        u_hyp = np.linspace(1, U_ke_max, 200)
        ax_out.plot(u_hyp, P_k_max/u_hyp*1000, 'purple', linestyle='-.', linewidth=1.5, label='Pк max')
        ax_out.plot(uke_q, ik_q, 'go', markersize=8, markeredgecolor='darkgreen', markeredgewidth=1.5, zorder=6, label='Рабочая точка')
        if ik_pp > 0:
            y_min, y_max = ik_q - ik_m, ik_q + ik_m
            ax_out.axhline(y=y_min, color='blue', linestyle=':', linewidth=1.5, alpha=0.7)
            ax_out.axhline(y=y_max, color='blue', linestyle=':', linewidth=1.5, alpha=0.7)
            ax_out.fill_betweenx([y_min, y_max], 0, U_ke_max, color='blue', alpha=0.1)
            ax_out.annotate('', xy=(uke_q*1.03, y_max), xytext=(uke_q*1.03, y_min), arrowprops=dict(arrowstyle='<->', color='blue', linewidth=1.5))
            ax_out.plot([], [], 'b:', linewidth=1.5, label=f'ΔIк (размах) = {ik_pp:.2f} мА')
        if uke_pp > 0:
            x_min, x_max = uke_q - uke_m, uke_q + uke_m
            ax_out.axvline(x=x_min, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
            ax_out.axvline(x=x_max, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
            ax_out.annotate('', xy=(x_max, ik_q*0.97), xytext=(x_min, ik_q*0.97), arrowprops=dict(arrowstyle='<->', color='orange', linewidth=1.5))
            ax_out.plot([], [], 'orange', linewidth=1.5, label=f'ΔUкэ (размах) = {uke_pp:.2f} В')
        ax_out.set_xlabel('Uкэ, В', fontweight='bold', fontsize=11)
        ax_out.set_ylabel('Iк, мА', fontweight='bold', fontsize=11)
        ax_out.set_title('Выходные характеристики', fontweight='bold', fontsize=13)
        ax_out.grid(True, alpha=0.3)
        ax_out.set_xlim(0, U_ke_max*1.05)
        ax_out.set_ylim(IK_MIN, IK_MAX)
        ax_out.legend(fontsize=8, loc='upper right', ncol=2)
        plt.tight_layout()
        plt.savefig('01_output_characteristics.png', dpi=300, bbox_inches='tight')
        print("✅ Сохранён: 01_output_characteristics.png")
        plt.close()
        
        # График 2: входная характеристика
        if ube_data is not None:
            fig_in = plt.figure(figsize=(8, 6))
            ax_in = fig_in.add_subplot(1, 1, 1)
            ax_in.plot(ib_in_data, ube_data, 'b-', linewidth=2.5, label='Входная ВАХ')
            ax_in.plot(ib_closest, ube_q, 'go', markersize=8, markeredgecolor='darkgreen', markeredgewidth=1.5, zorder=6, label='Рабочая точка')
            ube_min_val, ube_max_val = min(ube_data), max(ube_data)
            ube_range = ube_max_val - ube_min_val
            ube_pad = ube_range * 0.15 if ube_range > 0 else 0.1
            if ib_pp > 0:
                x_min, x_max = ib_closest - ib_m, ib_closest + ib_m
                ax_in.axvline(x=x_min, color='blue', linestyle=':', linewidth=1.5, alpha=0.7)
                ax_in.axvline(x=x_max, color='blue', linestyle=':', linewidth=1.5, alpha=0.7)
                ax_in.fill_between([x_min, x_max], min(ube_data), max(ube_data), color='blue', alpha=0.1)
                ax_in.annotate('', xy=(x_max, ube_q*1.03), xytext=(x_min, ube_q*1.03), arrowprops=dict(arrowstyle='<->', color='blue', linewidth=1.5))
                ax_in.plot([], [], 'b:', linewidth=1.5, label=f'ΔIб (размах) = {ib_pp:.2f} мкА')
            if ube_pp > 0:
                y_min, y_max = ube_q - ube_m, ube_q + ube_m
                ax_in.axhline(y=y_min, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
                ax_in.axhline(y=y_max, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
                ax_in.annotate('', xy=(ib_closest*1.05, y_max), xytext=(ib_closest*1.05, y_min), arrowprops=dict(arrowstyle='<->', color='orange', linewidth=1.5))
                ax_in.plot([], [], 'orange', linewidth=1.5, label=f'ΔUбэ (размах) = {ube_pp*1000:.2f} мВ')
            ax_in.set_xlabel('Iб, мкА', fontweight='bold', fontsize=11)
            ax_in.set_ylabel('Uбэ, В', fontweight='bold', fontsize=11)
            ax_in.set_title('Входная характеристика', fontweight='bold', fontsize=13)
            ax_in.grid(True, alpha=0.3)
            ax_in.set_xlim(IB_MIN, IB_MAX)
            ax_in.set_ylim(max(0, ube_min_val - ube_pad), ube_max_val + ube_pad)
            ax_in.legend(fontsize=9, loc='upper left')
            plt.tight_layout()
            plt.savefig('02_input_characteristic.png', dpi=300, bbox_inches='tight')
            print("✅ Сохранён: 02_input_characteristic.png")
            plt.close()
        
        # График 3: переходная характеристика
        if ib_trans is not None and len(ib_trans) > 0:
            fig_trans = plt.figure(figsize=(8, 6))
            ax_trans = fig_trans.add_subplot(1, 1, 1)
            ax_trans.plot(ib_trans, ik_trans, 'm-', linewidth=2.5, marker='o', markersize=4, label='Iк = f(Iб)')
            ax_trans.plot(ib_closest, ik_q, 'go', markersize=8, markeredgecolor='darkgreen', markeredgewidth=1.5, zorder=6, label='Рабочая точка')
            if ib_pp > 0 and ik_pp > 0:
                x_min, x_max = ib_closest - ib_m, ib_closest + ib_m
                y_min, y_max = ik_q - ik_m, ik_q + ik_m
                ax_trans.fill_between([x_min, x_max], y_min, y_max, color='green', alpha=0.15)
                ax_trans.axvline(x=x_min, color='blue', linestyle=':', linewidth=1, alpha=0.6)
                ax_trans.axvline(x=x_max, color='blue', linestyle=':', linewidth=1, alpha=0.6)
                ax_trans.axhline(y=y_min, color='blue', linestyle=':', linewidth=1, alpha=0.6)
                ax_trans.axhline(y=y_max, color='blue', linestyle=':', linewidth=1, alpha=0.6)
                ax_trans.plot([], [], 'g-', linewidth=3, alpha=0.15, label=f'Линейный участок: ΔIб={ib_pp:.2f}мкА, ΔIк={ik_pp:.2f}мА')
            ax_trans.set_xlabel('Iб, мкА', fontweight='bold', fontsize=11)
            ax_trans.set_ylabel('Iк, мА', fontweight='bold', fontsize=11)
            ax_trans.set_title('Переходная характеристика', fontweight='bold', fontsize=13)
            ax_trans.grid(True, alpha=0.3)
            ax_trans.set_xlim(IB_MIN, IB_MAX)
            ax_trans.set_ylim(IK_MIN, IK_MAX)
            ax_trans.legend(fontsize=9, loc='upper left')
            plt.tight_layout()
            plt.savefig('03_transfer_characteristic.png', dpi=300, bbox_inches='tight')
            print("✅ Сохранён: 03_transfer_characteristic.png")
            plt.close()
        
        # === ГРАФИК 4: ВРЕМЕННЫ́Е ДИАГРАММЫ ===
        plot_time_diagrams(ib_closest, ib_m, ube_q, ube_m, ik_q, ik_m, uke_q, uke_m)
        
        # === ВЫВОД АНАЛИТИЧЕСКИХ ВЫРАЖЕНИЙ ===
        print(f"\n📝 Аналитические выражения (синусоидальный входной сигнал):")
        print(f"   iБ(t)  = {ib_closest:.2f} + {ib_m:.2f}·sin(ωt)   [мкА]")
        print(f"   uБЭ(t) = {ube_q:.3f} + {ube_m:.3f}·sin(ωt)   [В]")
        print(f"   iК(t)  = {ik_q:.2f} + {ik_m:.2f}·sin(ωt)   [мА]")
        print(f"   uКЭ(t) = {uke_q:.2f} - {uke_m:.2f}·sin(ωt)   [В]  ⚡ инверсия фазы (ОЭ)")
        print(f"\n   где ω = 2πf, f — частота входного сигнала")
        
        # Экспорт результатов в файл
        with open('amplifier_results.txt', 'w', encoding='utf-8') as f:
            f.write("=== РЕЗУЛЬТАТЫ РАСЧЁТА УСИЛИТЕЛЬНОГО КАСКАДА ===\n\n")
            f.write("Режим покоя (рабочая точка):\n")
            f.write(f"  IБ0 = {ib_closest:.2f} мкА\n  IК0 = {ik_q:.2f} мА\n  UБЭ0 = {ube_q:.3f} В\n  UКЭ0 = {uke_q:.2f} В\n\n")
            f.write("Амплитуды переменного сигнала:\n")
            f.write(f"  IБм = {ib_m:.2f} мкА, размах ΔIБ = {ib_pp:.2f} мкА\n")
            f.write(f"  IКм = {ik_m:.2f} мА, размах ΔIК = {ik_pp:.2f} мА\n")
            f.write(f"  UБЭм = {ube_m:.3f} В, размах ΔUБЭ = {ube_pp*1000:.2f} мВ\n")
            f.write(f"  UКЭм = {uke_m:.2f} В, размах ΔUКЭ = {uke_pp:.2f} В\n\n")
            f.write("Элементы схемы:\n")
            for k, v in params.items():
                if k in ['R_E', 'C_E', 'R1', 'R2', 'R_B_calc', 'C_R1', 'E_K_corrected']:
                    unit = 'Ом' if 'R' in k else 'мкФ' if 'C' in k else 'В'
                    f.write(f"  {k} = {v:.2f} {unit}\n")
            f.write("\nПараметры усилителя:\n")
            for k in ['K_i', 'K_u', 'K_p', 'R_in_stage', 'R_out_stage', 'P_out', 'P_supply', 'eta']:
                unit = '' if k in ['K_i','K_u','K_p'] else 'Ом' if 'R' in k else 'Вт' if 'P' in k else '%'
                f.write(f"  {k} = {params[k]:.2f} {unit}\n")
        print("✅ Результаты сохранены в: amplifier_results.txt")

🔧 Расчёт усилительного каскада с ОЭ
✅ Загружено кривых: 11

🎯 Рабочая точка (режим покоя):
   IБ0 = 40.00 мкА
   IК0 = 14.99 мА
   UБЭ0 = 0.711 В
   UКЭ0 = 17.51 В

⚙️  Расчёт параметров усилителя...

📋 Элементы схемы:
   Rэ = 150 Ом  (падение ≈ 10% от Eк)
   Cэ = 70.7 мкФ  (fн = 100 Гц)
   R1 = 259.61 кОм
   R2 = 24.66 кОм
   Rб = 22.52 кОм
   Cр1 = 7.07 мкФ
   Eк (скорр.) = 42.2 В

📊 Параметры усилителя:
   β (оценка) = 100
   Kᵢ = 100.0  (по току)
   Kᵤ = -100.0  (по напряжению, знак «-» = инверсия)
   Kₚ = 10000  (по мощности)
   Rвх = 1.50 кОм
   Rвых ≈ 1.50 кОм
   Pвых = 1.69 мВт
   Pпит = 637.89 мВт
   η = 0.3%
   fверх ≈ 42.44 МГц
✅ Сохранён: 01_output_characteristics.png
✅ Сохранён: 02_input_characteristic.png
✅ Сохранён: 03_transfer_characteristic.png
✅ Сохранён: 04_time_diagrams.png

📝 Аналитические выражения (синусоидальный входной сигнал):
   iБ(t)  = 40.00 + 10.00·sin(ωt)   [мкА]
   uБЭ(t) = 0.711 + 0.008·sin(ωt)   [В]
   iК(t)  = 14.99 + 3.09·sin(ωt)   [мА]
   uКЭ(t) = 1